## 1. Importing libraries and dataset 

In [12]:
import csv

def load_csv_dataset(filename, dialect='excel', delimiter = ','):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, dialect=dialect, delimiter=delimiter)
        return [row for row in reader]

    
def save_csv_dataset(filename, data, header=None):
    header = header if header else list(data[0].keys())
    with open(filename, 'w', encoding="utf-8") as f:
        writer = csv.DictWriter(f,fieldnames=header)
        writer.writeheader()
        for d in data:
            writer.writerow(d)

def extract_respondents_id(code: str):
    """
    A function to link the practitioners to the requirements considering the indicators in the code that was used to elicitate de requirement
        
        Parameters
        ----------
        code: str
            the code (generated in open coding) that represents a requirement

    """

    # The symbol "○" indicates the begining of a code, e.g., â—‹ 5-the bot is already providing too much info
    # The symbol "¶" indicates a quotation, e.g., 
    # 5:5 ¶ 6, in s1_r25 
    # Keeping track of tasks and specially technical debts can be hard sometimes,
    # In a quoatation, it is possible to find the respondent id, e.g., s1_r25, which is locate in the end of the line

    respondents = []
    with open("../data/codes-report-exported-from-atlasti.txt") as f:
        all_lines = f.readlines()
        
        # checks if the line contains a character that indicates code description
        # "i" represents the line index, while "l" is the line content
        lines_with_code_description = [line_index for line_index, line_content in enumerate(all_lines) if "○" in line_content]

        # iterates over the line indexes
        for current_index_of_code_description_array, current_number_of_line_with_code_description in enumerate(lines_with_code_description):
            if code in all_lines[current_number_of_line_with_code_description]:
                if current_index_of_code_description_array == len(lines_with_code_description) - 1: # true if the script is checking the last index in the lines_with_code_description array
                    next_line_with_code_description = len(all_lines)
                else:
                    next_line_with_code_description = lines_with_code_description[current_index_of_code_description_array + 1]
                
                # sets the line_counter on the first line after the code descriptio
                line_index_counter = current_number_of_line_with_code_description + 1

                # check lines between one code description and the next one
                while line_index_counter < next_line_with_code_description:

                    # checks if the line contains a character that indicates quotation
                    if "¶" in all_lines[line_index_counter]:
                        # splits the line in each space, e.g., ['', '', '72:1', 'Â¶', '2,', 'in', 's4_r5\n']
                        # takes the last element of the array, which represents the respondent id
                        # removes the line break character character ("\n")
                        respondents.append(all_lines[line_index_counter].split(" ")[-1].replace("\n", ""))

                    # sets the line counter on the next line
                    line_index_counter += 1
                return respondents

requirements_dataset = load_csv_dataset("../data/requirements-dataset.csv")
respondents_dataset = load_csv_dataset("../data/respondents-dataset.csv", delimiter=';')
dataset = []

for r in requirements_dataset:
    respondents_per_code = list(set(extract_respondents_id(r['code'])))
    
    for respondent in respondents_per_code:
        try:
            id = [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0]
        except:
            continue
        dataset.append({'scenario': r['scenario'], 
                        'category': r['category'], 
                        'complete-requirement': r['complete-requirement'], 
                        'code': r['code'],
                        'respondent-id':respondent,
                        'respondent-role': [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-experience':[r1['experience'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-education':[r1['education'] for r1 in respondents_dataset if r1['id'] == respondent][0]})
        
save_csv_dataset("../data/requirements-and-respondents.csv", dataset)

### 2. Data analysis

### 2.1. Number of requirements per per scenario

In [13]:
requirements_dataset

,scenario,category,complete-requirement,code,respondent-id,respondent-role,respondent-experience,respondent-education
0,s1,information to be provided/detailed information,The issue labeling bot may notify practitioner...,1-notifications are useful,s1_r4,Software Developer,1 - 5 years,Bachelor
1,s1,information to be provided/detailed information,The issue labeling bot may notify practitioner...,1-notifications are useful,s1_r23,Software Developer,1 - 5 years,Bachelor
2,s1,information to be provided/detailed information,The issue labeling bot may notify practitioner...,1-notifications are useful,s1_r19,Software Architect/Designer,more than 10 years,PhD
3,s1,information to be provided/detailed information,The issue labeling bot may notify practitioner...,1-notifications are useful,s1_r15,Software Architect/Designer,1 - 5 years,Master
4,s1,information to be provided/detailed information,The issue labeling bot may notify practitioner...,1-notifications are useful,s1_r25,Software Developer,6 - 10 years,Bachelor
...,...,...,...,...,...,...,...,...
107,s5,tool usage/tool execution/workflow execution,The code review bot may notify the commit auth...,5-notify commit author,s5_r13,Software Developer,more than 10 years,Bachelor
108,s5,information to be provided/detailed information,The code review bot may include links to the c...,5-link to facilitate navigation,s5_r21,Software Developer,1 - 5 years,Bachelor
109,s5,tool usage/customization,The code review bot may provide a command mark...,5-command:mark code smells as false positives,s5_r20,Engineering Manager,more than 10 years,Unfinished bachelor
110,s5,information to be provided/grouped information,The code review bot may provide long-term impa...,5-long-term impact,s5_r16,System Architect,more than 10 years,Bachelor


In [14]:
requirements_dataset.groupby(['scenario','complete-requirement']).count()

category  code  \
scenario complete-requirement                                                 
s1       The issue labeling bot may enable practitioners...         4     4   
         The issue labeling bot may notify practitioners...         9     9   
         The issue labeling bot may provide a customizat...         3     3   
         The issue labeling bot may provide a customizat...         4     4   
         The issue labeling bot may provide a customizat...         1     1   
         The issue labeling bot may provide a customizat...         4     4   
         The issue labeling bot may provide a customizat...         2     2   
         The issue labeling bot may send notifications b...         1     1   
         The issue labeling bot may send notifications b...         2     2   
s2       The dashboard may contain a dedicated widget fo...         8     8   
         The dashboard may present a list of issues grou...         1     1   
         The dashboard may present a list of smells grou...         6     6   
         The dashboard may present a list of smells grou...         4     4   
         The dashboard may present test code coverage co...         1     1   
         The dashboard may present the evolution of the ...         2     2   
         The dashboard may present the time since an iss...         5     5   
         The dashboard may provide a customization optio...         2     2   
         The dedicated technical debt dashboard may pres...         1     1   
         The dedicated technical debt dashboard may pres...         3     3   
         The dedicated technical debt dashboard may pres...         2     2   
         The dedicated technical debt dashboard may pres...         2     2   
s3       The refactoring plugin may enable manually-trgg...         5     5   
         The refactoring plugin may inform the impact of...         1     1   
         The refactoring plugin may present refactoring ...         1     1   
         The refactoring plugin may provide refactoring ...         4     4   
s4       The ci-script may include the amount of code sm...         1     1   
         The ci-script may include the metric that trigg...         1     1   
         The ci-script may present qaulity metrics using...         1     1   
         The ci-script may send a report containig the e...         3     3   
         The ci-script may send notification using insta...         6     6   
         The ci-script may send notification using maili...         1     1   
         The ci-script may send notifications to practit...         2     2   
         The ci-script may send notifications to practit...         3     3   
         The ci-script may send notifications to practit...         1     1   
         The ci-script may send the summary of the data ...         2     2   
s5       The code review bot may include links to the co...         1     1   
         The code review bot may notify the commit autho...         1     1   
         The code review bot may notify the contributors...         1     1   
         The code review bot may notify the tech lead/ar...         3     3   
         The code review bot may provide a command for i...         1     1   
         The code review bot may provide a command mark ...         1     1   
         The code review bot may provide additional info...         2     2   
         The code review bot may provide long-term impac...         1     1   
         The code review bot may provide the feature or ...         1     1   
         The code review bot may provide the number of c...         1     1   

                                                             respondent-id  \
scenario complete-requirement                                                
s1       The issue labeling bot may enable practitioners...              4   
         The issue labeling bot may notify practitioners...              9   
         The issue labeling bot may

### 2.2. Requirements per category

In [88]:
r = requirements_dataset[['category', 'respondent-id', 'respondent-education']].groupby(["category","respondent-education",'respondent-id']).count()
r.to_csv('education-per-category')

In [89]:
r = requirements_dataset[['category', 'respondent-id', 'respondent-role']].groupby(["category","respondent-role",'respondent-id']).count()
r.to_csv('role-per-category.csv')

In [90]:
r = requirements_dataset[['category', 'respondent-id', 'respondent-experience']].groupby(["category","respondent-experience",'respondent-id']).count()
r.to_csv('experience-per-category.csv')

In [57]:
import csv
def load_csv_dataset(filename, dialect='excel', delimiter = ','):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, dialect=dialect, delimiter=delimiter)
        return [row for row in reader]

respondents_dataset = load_csv_dataset("../data/respondents-dataset.csv", delimiter=';')

r['education'] = ''

In [59]:
for i in r.index:
    id = i[1]
    education = [x for x in respondents_dataset if  x['id'] == id][0]['education']
    r['education'][i] = education

/tmp/ipykernel_86/1015633255.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r['education'][i] = education
/tmp/ipykernel_86/1015633255.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r['education'][i] = education
/tmp/ipykernel_86/1015633255.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r['education'][i] = education
/tmp/ipykernel_86/1015633255.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

In [60]:
r.group

code role  \
category                                        respondent-id              
information to be provided/detailed information s1_r14            1        
                                                s1_r15            1        
                                                s1_r17            1        
                                                s1_r19            1        
                                                s1_r21            1        
...                                                             ...  ...   
tool usage/tool interface                       s4_r2             1        
                                                s4_r3             1        
                                                s4_r4             1        
                                                s4_r5             1        
                                                s4_r6             1        

                                                              education  
category                                        respondent-id            
information to be provided/detailed information s1_r14           Master  
                                                s1_r15           Master  
                                                s1_r17              PhD  
                                                s1_r19              PhD  
                                                s1_r21         Bachelor  
...                                                                 ...  
tool usage/tool interface                       s4_r2          Bachelor  
                                                s4_r3            Master  
                                                s4_r4               PhD  
                                                s4_r5            Master  
                                                s4_r6               PhD  

[96 rows x 3 columns]

### 2.3. Number of respondents per requirement

In [16]:
requirements_dataset[['respondent-id', 'complete-requirement', 'code', 'category']].groupby(["complete-requirement", "code", 'category']).count().sort_values('respondent-id', ascending=False)

,,,respondent-id
complete-requirement,code,category,
The issue labeling bot may notify practitioners about issues containing techincal debt.,1-notifications are useful,information to be provided/detailed information,9
The dashboard may contain a dedicated widget for technical debt,2-widget is useful,tool usage/tool interface,8
The dashboard may present a list of smells grouped by package,2-group code smells per package,information to be provided/detailed information,6
The ci-script may send notification using instant message services,4-instant message could be used for notificaitons,tool usage/tool interface,6
The refactoring plugin may enable manually-trggered execution,3-manual trigger is the preference,tool usage/tool execution/manual execution,5
The dashboard may present the time since an issue was open.,2-issue age,information to be provided/detailed information,5
The issue labeling bot may provide a customization option to configure to which practitioners notifications will be sent.,1-notifications should be sent to interested people only,tool usage/tool execution/threshold execution,4
The issue labeling bot may provide a customization option to configure notifications considering the lable assigned to an issue.,1-customization options are necessary,tool usage/tool execution/threshold execution,4
The issue labeling bot may enable practitioners to choose whether to assign a label to an issue,1-customization options are necessary,tool usage/customization,4


## Chi-square Test

In [6]:
import pandas as pd
from scipy import stats

#### Test Category vs Charachtericts

In [18]:
#Test Role
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-role'])
stats.chi2_contingency(crosstab)

Chi2ContingencyResult(statistic=35.14237916041838, pvalue=0.5091986644155265, dof=36, expected_freq=array([[ 4.6875    ,  4.375     ,  0.625     ,  6.25      , 16.875     ,
         0.9375    ,  1.25      ],
       [ 2.00892857,  1.875     ,  0.26785714,  2.67857143,  7.23214286,
         0.40178571,  0.53571429],
       [ 1.07142857,  1.        ,  0.14285714,  1.42857143,  3.85714286,
         0.21428571,  0.28571429],
       [ 0.66964286,  0.625     ,  0.08928571,  0.89285714,  2.41071429,
         0.13392857,  0.17857143],
       [ 2.27678571,  2.125     ,  0.30357143,  3.03571429,  8.19642857,
         0.45535714,  0.60714286],
       [ 1.47321429,  1.375     ,  0.19642857,  1.96428571,  5.30357143,
         0.29464286,  0.39285714],
       [ 2.8125    ,  2.625     ,  0.375     ,  3.75      , 10.125     ,
         0.5625    ,  0.75      ]]))

In [19]:
#Test Experience
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-experience'])
stats.chi2_contingency(crosstab)

Chi2ContingencyResult(statistic=12.101028350057339, pvalue=0.4376004294022825, dof=12, expected_freq=array([[11.875     ,  3.4375    , 19.6875    ],
       [ 5.08928571,  1.47321429,  8.4375    ],
       [ 2.71428571,  0.78571429,  4.5       ],
       [ 1.69642857,  0.49107143,  2.8125    ],
       [ 5.76785714,  1.66964286,  9.5625    ],
       [ 3.73214286,  1.08035714,  6.1875    ],
       [ 7.125     ,  2.0625    , 11.8125    ]]))

In [20]:
#Test Experience
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-education'])
stats.chi2_contingency(crosstab)

Chi2ContingencyResult(statistic=15.054030751880937, pvalue=0.6582538568421055, dof=18, expected_freq=array([[12.1875    ,  9.6875    ,  7.1875    ,  5.9375    ],
       [ 5.22321429,  4.15178571,  3.08035714,  2.54464286],
       [ 2.78571429,  2.21428571,  1.64285714,  1.35714286],
       [ 1.74107143,  1.38392857,  1.02678571,  0.84821429],
       [ 5.91964286,  4.70535714,  3.49107143,  2.88392857],
       [ 3.83035714,  3.04464286,  2.25892857,  1.86607143],
       [ 7.3125    ,  5.8125    ,  4.3125    ,  3.5625    ]]))

In [21]:
crosstab

respondent-education,Bachelor,Master,PhD,Unfinished bachelor
category,,,,
information to be provided/detailed information,14,10,8,3
information to be provided/grouped information,3,6,4,2
tool usage/customization,4,1,0,3
tool usage/tool execution/manual execution,3,1,1,0
tool usage/tool execution/threshold execution,6,4,2,5
tool usage/tool execution/workflow execution,4,3,3,1
tool usage/tool interface,5,6,5,5


#### Test Requirements vs Charachtericts

In [11]:
#Test Role
crosstab = pd.crosstab(requirements_dataset['complete-requirement'], requirements_dataset['respondent-role'])
stats.chi2_contingency(crosstab)

Chi2ContingencyResult(statistic=263.47818181818184, pvalue=0.6003273411826784, dof=270, expected_freq=array([[0.13157895, 0.12280702, 0.01754386, 0.18421053, 0.48245614,
        0.02631579, 0.03508772],
       [0.13157895, 0.12280702, 0.01754386, 0.18421053, 0.48245614,
        0.02631579, 0.03508772],
       [0.13157895, 0.12280702, 0.01754386, 0.18421053, 0.48245614,
        0.02631579, 0.03508772],
       [0.39473684, 0.36842105, 0.05263158, 0.55263158, 1.44736842,
        0.07894737, 0.10526316],
       [0.78947368, 0.73684211, 0.10526316, 1.10526316, 2.89473684,
        0.15789474, 0.21052632],
       [0.13157895, 0.12280702, 0.01754386, 0.18421053, 0.48245614,
        0.02631579, 0.03508772],
       [0.26315789, 0.24561404, 0.03508772, 0.36842105, 0.96491228,
        0.05263158, 0.07017544],
       [0.39473684, 0.36842105, 0.05263158, 0.55263158, 1.44736842,
        0.07894737, 0.10526316],
       [0.13157895, 0.12280702, 0.01754386, 0.18421053, 0.48245614,
        0.02631579, 0.